ARTI308 - Machine Learning

# Project Notebook 3: Feature Engineering
## Heart Disease Prediction

**Feature engineering** means creating new, more informative features from the existing ones, using knowledge about the problem (here: medicine).

In this notebook we create **two new features**:
1. `age_group`: patients grouped into age ranges.
2. `maxhr_pct`: the maximum heart rate reached, as a fraction of the maximum expected for the patient's age.

### Why do it here and not inside the Pipeline?
Both features are calculated **row by row** from values of the same patient. They do not learn anything from the dataset (no mean, median, min or max), so they cannot cause data leakage, and it is safe to create them **before** the train/test split.
(Imputation and scaling are different: they learn from the data, so they stay inside the Pipeline.)

Run notebooks 01 and 02 first (notebook 02 creates `data/processed/heart_clean.csv`).

## 1. Setup and load data

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/processed/heart_clean.csv")
target_col = "HeartDisease"

print("Shape:", df.shape)
df.head()

## 2. Feature 1: `age_group`
Heart-disease risk is not a straight line with age: it jumps at certain stages of life. Grouping `Age` into ranges lets the model learn a separate effect for each group.

We use four groups: under 40, 40-49, 50-59, and 60 or more.

In [ ]:
df["age_group"] = pd.cut(df["Age"], bins=[0, 40, 50, 60, 200],
                         labels=["under_40", "40_49", "50_59", "60_plus"]).astype(str)

order = ["under_40", "40_49", "50_59", "60_plus"]
df["age_group"].value_counts().reindex(order)

**Observation:**

*(Write what you notice after running the cell above.)*

Does the heart-disease rate change between the groups?

In [ ]:
rate = df.groupby("age_group")[target_col].mean().reindex(order)

plt.figure(figsize=(6, 4))
plt.bar(rate.index, rate.values, color="steelblue")
plt.title("Heart disease rate by age group")
plt.ylabel("Proportion with heart disease")
plt.ylim(0, 1)
plt.show()

rate.round(3)

**Observation:**

*(Write what you notice after running the cell above.)*

## 3. Feature 2: `maxhr_pct`
A common rule of thumb says the **maximum heart rate** a person can reach is about `220 - Age`.
A 30-year-old and a 70-year-old with the same `MaxHR` (say 140) are very different: 140 is normal for the older patient but low for the younger one.

So we divide the measured `MaxHR` by the expected maximum for that age:

`maxhr_pct = MaxHR / (220 - Age)`

A value near 1 means the patient reached close to the expected maximum; a low value means the heart did not reach what is expected for the age.

In [ ]:
df["maxhr_pct"] = df["MaxHR"] / (220 - df["Age"])

df[["Age", "MaxHR", "maxhr_pct"]].head()

In [ ]:
df["maxhr_pct"].describe().round(3)

**Observation:**

*(Write what you notice after running the cell above.)*

Compare the new feature with the original `MaxHR` for patients with and without heart disease.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(x=target_col, y="MaxHR", data=df, ax=axes[0])
axes[0].set_title("MaxHR (original)")
sns.boxplot(x=target_col, y="maxhr_pct", data=df, ax=axes[1])
axes[1].set_title("maxhr_pct (new)")
plt.tight_layout()
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

## 4. Are the new features useful?
A quick check: the correlation of each numeric feature with the target. This is only an indication (it measures linear relationships); the real test is the model performance in the next notebooks.

In [ ]:
num = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak", "FastingBS", "maxhr_pct"]
corr_target = df[num + [target_col]].corr()[target_col].drop(target_col).sort_values()

plt.figure(figsize=(7, 4))
colors = ["tomato" if f == "maxhr_pct" else "steelblue" for f in corr_target.index]
plt.barh(corr_target.index, corr_target.values, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlation with HeartDisease (new feature in red)")
plt.show()

corr_target.round(3)

**Observation:**

*(Write what you notice after running the cell above.)*

### Redundancy check
`maxhr_pct` is built from `MaxHR` and `Age`, so it will be correlated with them. That is expected. Logistic Regression can handle this, and tree-based models are not affected by it.

In [ ]:
df[["Age", "MaxHR", "maxhr_pct"]].corr().round(2)

**Observation:**

*(Write what you notice after running the cell above.)*

## 5. What we do NOT do here
- **Encoding** the text columns (`Sex`, `ChestPainType`, ...) is done inside the Pipeline with `OneHotEncoder`.
- **Imputation and scaling** are done inside the Pipeline, after the split (to avoid data leakage).
- `age_group` is saved as text, so it will also be one-hot encoded by the Pipeline.

## 6. Save the dataset with the new features
The next notebooks (04 to 07) read this file, so all four models use exactly the same features.
Missing values are still `NaN`; they are filled later inside the Pipeline.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/heart_features.csv", index=False)

print("Saved:", df.shape)
print("New columns:", [c for c in df.columns if c in ("age_group", "maxhr_pct")])
print("Missing values kept as NaN:")
print(df.isna().sum()[df.isna().sum() > 0])

---
### Task
Create one more feature of your own, for example `high_bp = (RestingBP >= 140)` (use `.astype(int)`), and check whether the heart-disease rate differs between the two groups. Would you keep it? Explain why.